In [1]:
from os.path import join

from data.pipeline import build_vocabulary
from data.vocabulary import load_vocabulary
from configs import configs

from train import train_word2vec

import pickle
import zstandard as zstd

import torch
from word2vec import Word2Vec

import numpy as np

import gc

from embed import Embedder

## Build Sentence Corpus and Vocabulary

In [ ]:
file_path = r"C:\Users\arka\Documents\summer 26\llm\embeddings\datasets\kaggle\simple_english_wikipedia.txt"
context_window_size = 2 or configs['samples']['CONTEXT_WINDOW_SIZE']
min_frequency = 1 or configs['samples']['MIN_TERM_FREQUENCY']
alpha = configs['samples']['ALPHA']
vocabulary_dir = configs['cache']['VOCABULARY_PATH']
samples_dir = configs['cache']['SAMPLES_PATH']

In [ ]:
build_vocabulary(file_path, min_frequency, alpha, vocabulary_dir)

## Load Vocabulary

In [5]:
vocabulary = load_vocabulary(configs['cache']['VOCABULARY_PATH'])
print("Vocabulary Size:", len(vocabulary))

Vocabulary Size: 615001


## Train Word2Vec model

In [ ]:
model = train_word2vec()

## Save and Index embeddings

In [7]:
save_path = join(configs['cache']['EMBEDDINGS_PATH'], "word2vec_embeddings_128d.pt")
embedding_dim = configs['training']['EMBEDDING_DIM']

state_dict = torch.load(save_path, weights_only=True)

model = Word2Vec(
    vocab_size=len(vocabulary),
    embedding_dim=embedding_dim,
)
model.load_state_dict(state_dict)

input_embeddings = model.input_embeddings.weight.detach().cpu().numpy()
output_embeddings = model.output_embeddings.weight.detach().cpu().numpy()

np.save(join(configs['cache']['EMBEDDINGS_PATH'], "input_embeddings.npy"), input_embeddings)
np.save(join(configs['cache']['EMBEDDINGS_PATH'], "output_embeddings.npy"), output_embeddings)

del input_embeddings, output_embeddings, state_dict
gc.collect()

1491

In [8]:
embed = Embedder(
    vocabulary=vocabulary, 
    embedding_path=join(configs['cache']['EMBEDDINGS_PATH'], "input_embeddings.npy")
)

In [9]:
words = ['King', 'Queen', 'Man', 'Woman']

embeddings = embed(words)

In [10]:
def cosine_similarity(u, v):
    return np.arccos(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v)))

In [11]:
king, queen, man, woman = embeddings

diff1 = king - queen
diff2 = man - woman

cosine_similarity(diff1, diff2)

np.float32(1.4457489)

In [12]:
embed.build_index(
    embedding_path=join(configs['cache']['EMBEDDINGS_PATH'], "input_embeddings.npy"),
    index_path=join(configs['cache']['INDEX_PATH'], "input_embeddings.index")
)

2026-06-29 11:17:26,157 INFO word2vec: Total vectors indexed: 615001


In [19]:
embed.nearest("Good")

['Good',
 'My',
 'Life',
 'Our',
 'titled',
 'Only',
 'Will',
 'Young',
 'Baby',
 'Long']